# Online and Batch Predictions on Vertex AI

In the previous lab, we promoted the challenger to 100% traffic and undeployed the old champion.
For this lab, the challenger has been re-deployed alongside the champion with a 90/10 traffic split.

In this notebook we:
1. Connect to our endpoint
2. (Optional) Re-deploy the challenger if you only have one model
3. Online prediction - single instance
4. Online prediction - multiple instances
5. Traffic split demo - A/B testing
6. Batch prediction
7. Read batch results

In [ ]:
from google.cloud import aiplatform
from google.cloud import storage
import json

In [ ]:
PROJECT_ID = "YOUR_PROJECT_ID"      # Replace with your GCP project ID
REGION = "us-central1"
BUCKET_NAME = "YOUR_BUCKET_NAME"    # Replace with your GCS bucket name

aiplatform.init(project=PROJECT_ID, location=REGION)

## 1. Connect to the Existing Endpoint

In [ ]:
ENDPOINT_DISPLAY_NAME = "bikeshare-prediction-endpoint"

endpoints = aiplatform.Endpoint.list(filter=f'display_name="{ENDPOINT_DISPLAY_NAME}"')
endpoint = endpoints[0]
print(f"Endpoint: {endpoint.display_name}")

for dm in endpoint.list_models():
    print(f"  Deployed: {dm.display_name} (ID: {dm.id})")
print(f"Traffic split: {endpoint.traffic_split}")

## 2. (Optional) Re-deploy the Challenger
In the previous lab we undeployed one of the models. If you want to test the traffic split
with two models, run the cell below. If you already have both models deployed, skip this cell.

In [ ]:
# OPTIONAL - Only run this if you have a single model deployed.
# Skip this cell if both champion and challenger are already on the endpoint.

MODEL_DISPLAY_NAME = "bikeshare-model"

registry = aiplatform.models.ModelRegistry(
    model=aiplatform.Model.list(filter=f'display_name="{MODEL_DISPLAY_NAME}"')[0].resource_name
)

challenger_model = registry.get_model(version="challenger")

endpoint.deploy(
    model=challenger_model,
    deployed_model_display_name="challenger-deployment",
    machine_type="n1-standard-4",
    min_replica_count=1,
    max_replica_count=1,
    traffic_percentage=10,
)

print(f"Challenger deployed (version {challenger_model.version_id})")
print(f"Traffic split: {endpoint.traffic_split}")

## 3. Online Prediction - Single Instance
Send one instance to the endpoint. The response includes `deployed_model_id` so you can see which model served the request.

In [ ]:
instance = [0.24, 0.81] + [0] * 43 + [1] + [0] * 4  # 50 features

response = endpoint.predict(instances=[instance])

print(f"Predicted log-count: {response.predictions[0]:.4f}")
print(f"Served by deployed model ID: {response.deployed_model_id}")

## 4. Online Prediction - Multiple Instances
Send several instances in one request. Vertex AI batches them internally.

In [ ]:
instances = [
    [0.24, 0.81] + [0] * 43 + [1] + [0] * 4,    # cool day, low humidity
    [0.80, 0.27] + [0, 1] + [0] * 42 + [1] + [0] * 3,  # hot day, season 3
    [0.50, 0.50] + [0] * 43 + [1] + [0] * 4,    # moderate conditions
]

response = endpoint.predict(instances=instances)

for i, pred in enumerate(response.predictions):
    print(f"Instance {i+1}: predicted log-count = {pred:.4f}")
print(f"\nServed by deployed model ID: {response.deployed_model_id}")

## 5. Traffic Split Demo - A/B Testing
Send 20 requests and see how they get routed across both models.

In [ ]:
from collections import Counter

model_counts = Counter()
instance = [0.24, 0.81] + [0] * 43 + [1] + [0] * 4

for _ in range(100):
    resp = endpoint.predict(instances=[instance])
    model_counts[resp.deployed_model_id] += 1

print("Requests per deployed model:")
for model_id, count in model_counts.items():
    print(f"  Model ID {model_id}: {count}/100 requests")
print("\nYou should see roughly 90/10 split across the two models.")

## 6. Batch Prediction
For large datasets, use batch prediction. It runs asynchronously against a CSV in GCS.

Batch prediction runs against a specific model version, not the endpoint traffic split. We'll use the champion.

In [ ]:
MODEL_DISPLAY_NAME = "bikeshare-model"

registry = aiplatform.models.ModelRegistry(
    model=aiplatform.Model.list(filter=f'display_name="{MODEL_DISPLAY_NAME}"')[0].resource_name
)
champion_model = registry.get_model(version="champion")
print(f"Running batch prediction with champion (version {champion_model.version_id})")
print(f"Note: batch prediction runs against a specific model version, not the endpoint traffic split.")

In [ ]:
GCS_INPUT = f"gs://{BUCKET_NAME}/bike-share/batch-small.csv"
GCS_OUTPUT = f"gs://{BUCKET_NAME}/bikeshare-batch-output"

batch_job = champion_model.batch_predict(
    job_display_name="bikeshare-batch-predict",
    gcs_source=GCS_INPUT,
    gcs_destination_prefix=GCS_OUTPUT,
    instances_format="csv",
    predictions_format="jsonl",
    machine_type="n1-standard-4",
    starting_replica_count=1,
    max_replica_count=1,
    sync=False,
)

print(f"Batch job submitted: {batch_job.display_name}")
print("Waiting for completion...")
batch_job.wait()
print(f"Batch job completed")
print(f"Output: {batch_job.output_info.gcs_output_directory}")

## Optional: Recover batch_job after kernel restart
If your kernel died during the batch prediction, run this cell to reconnect to the completed job
instead of re-running everything from the top.

In [ ]:
from google.cloud import aiplatform
from google.cloud import storage
import json

PROJECT_ID = "YOUR_PROJECT_ID"      # Replace with your GCP project ID
REGION = "us-central1"
BUCKET_NAME = "YOUR_BUCKET_NAME"    # Replace with your GCS bucket name
aiplatform.init(project=PROJECT_ID, location=REGION)

jobs = aiplatform.BatchPredictionJob.list(filter='display_name="bikeshare-batch-predict"')
batch_job = jobs[0]
print(f"Output: {batch_job.output_info.gcs_output_directory}")

## 7. Read Batch Prediction Results
Results are written as JSONL files in GCS. Each line is a JSON object with the prediction.

In [ ]:
output_dir = batch_job.output_info.gcs_output_directory
prefix = output_dir.replace(f"gs://{BUCKET_NAME}/", "")

storage_client = storage.Client()
bucket = storage_client.bucket(BUCKET_NAME)
blobs = list(bucket.list_blobs(prefix=prefix))

print(f"Output files: {len(blobs)}")
for blob in blobs:
    if blob.name.endswith(".jsonl"):
        content = blob.download_as_text()
        lines = content.strip().split("\n")
        print(f"\nFile: {blob.name}")
        print(f"Total predictions: {len(lines)}")
        print("\nFirst 5 predictions:")
        for line in lines[:5]:
            result = json.loads(line)
            print(f"  Prediction: {result['prediction']:.4f}")